In [ ]:
import functools as f
from datetime import datetime

from sklearn.cluster import SpectralClustering
from numpy.typing import NDArray
import numpy as np

from common import grid_search, load_data, cross_validate

In [ ]:
def get_clusters(
    adj_matrix: NDArray[np.float64 | np.int32], hyperparameters: dict, seed=456
):
    clustering = SpectralClustering(
        n_clusters=hyperparameters["n_clusters"],
        assign_labels=hyperparameters["strategy"],
        random_state=seed,
    ).fit(adj_matrix)

    return clustering.labels_

In [ ]:
def generate_hyperparameters_for_sc(max_number_clusters: int):
    combinations = []

    for strategy in ["kmeans", "discretize", "cluster_qr"]:
        combinations.append(
            {
                "strategy": strategy,
            }
        )

    return combinations

In [ ]:
method = "sc"
dataset = "dwug_es"
path_to_gold_data = "./gold-data-es.csv" 
path_to_data= f"./deepmistake_model_es.csv"
max_number_clusters = 5
model = "deepmistake"

In [ ]:
metadata = {
    "method": method,
    "dataset": dataset,
    "path_to_data": path_to_data,
    "path_to_gold_data": path_to_gold_data,
    "fill_diagonal": True,
    "normalize": True,
    "model": model,
    "use_threshold": True,
}

In [ ]:
start_time = datetime.now()

# grid_search(
#     f.partial(load_data, path_to_data),
#     get_clusters,
#     generate_hyperparameters_for_sc(max_number_clusters=max_number_clusters),
#     metadata=metadata,
# )

print(f"Elapsed time: {datetime.now() - start_time}")

## Cross-validation experiments

In [ ]:
gold_dir = "./dwug_es_cleaned/clusters"

In [ ]:
start_time = datetime.now()

for selection_method in ["calinski_harabasz", "eigengap"]:
    metadata["cluster_selection_method"] = selection_method

    cv_summary = cross_validate(
        get_clusters,
        generate_hyperparameters_for_sc(max_number_clusters=max_number_clusters),
        metadata=metadata,
        gold_dir=gold_dir,
        k=5,
    )
    
    print(f"\nProtocol 1 (ARI driven): ")
    print(f"  avg test ARI: {cv_summary['protocol_ari']['avg_test_ari']:.4f}")
    print(f"  avg test LSCD: {cv_summary['protocol_ari']['avg_test_lscd']:.4f}")
    print(f"\nProtocol 2 (LSCD Driven):")
    print(f"  avg test LSCD: {cv_summary['protocol_lscd']['avg_test_lscd']:.4f}")
    print(f"  avg test ARI: {cv_summary['protocol_lscd']['avg_test_ari']:.4f}")

print(f"Elapsed time: {datetime.now() - start_time}")